# Image Processing Pipeline Notebook

In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def ensure_gray(image):
    if image is None:
        raise ValueError('Expected image input')
    if image.ndim == 2:
        return image
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

def ensure_bgr(image):
    if image is None:
        raise ValueError('Expected image input')
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
    return image

def odd_ksize(value, minimum=1):
    k = int(value)
    k = max(int(minimum), k)
    if k % 2 == 0:
        k += 1
    return k

def gamma_correct(image, gamma):
    gamma = max(0.01, float(gamma))
    table = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(image, table)

def show(image, title='Preview', figsize=(10, 6), cmap=None):
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap=cmap or 'gray')
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
# Cells Input (ImageInput)
image_path = '/Users/oren/code/cme466/class_example_code/imgs/cells.jpg'
cells_input = cv2.imread(str(Path(image_path).expanduser()))
if cells_input is None:
    raise FileNotFoundError(f'Unable to load image: {image_path}')

show(cells_input, title='Cells Input - image')

In [ ]:
# Crop 30px Border (ImageCrop)
_crop_img = ensure_bgr(cells_input)
_h, _w = _crop_img.shape[:2]
x = max(0, min(_w - 1, 30))
y = max(0, min(_h - 1, 30))
crop_w = 444
crop_h = 271
if crop_w <= 0:
    crop_w = _w - x
if crop_h <= 0:
    crop_h = _h - y
x2 = min(_w, x + max(1, crop_w))
y2 = min(_h, y + max(1, crop_h))
if x2 <= x or y2 <= y:
    raise ValueError('Invalid crop rectangle')
crop_30px_border_image = _crop_img[y:y2, x:x2].copy()
crop_30px_border_meta = {
    'crop_box': {'x': x, 'y': y, 'width': x2 - x, 'height': y2 - y},
    'source_size': {'width': _w, 'height': _h},
}

show(crop_30px_border_image, title='Crop 30px Border - image')

In [ ]:
# Branch RGB/Mask (SplitImage)
_split_img = ensure_bgr(crop_30px_border_image)
branch_rgb_mask_image_a = _split_img.copy()
branch_rgb_mask_image_b = _split_img.copy()

show(branch_rgb_mask_image_a, title='Branch RGB/Mask - image_a')
show(branch_rgb_mask_image_b, title='Branch RGB/Mask - image_b')

In [ ]:
# Gray (GrayConvert)
gray_mask = ensure_gray(branch_rgb_mask_image_a)
gray_image = cv2.cvtColor(gray_mask, cv2.COLOR_GRAY2BGR)

show(gray_image, title='Gray - image')
show(gray_mask, title='Gray - mask', cmap='gray')

In [ ]:
# Gaussian Blur k=7 (DenoiseBlur)
_blur_img = ensure_bgr(gray_image)
blur_method = 'gaussian'
kx = odd_ksize(7, 1)
ky = odd_ksize(7, 1)
sigma_x = 0.0
sigma_y = 0.0
gaussian_blur_k_7 = cv2.GaussianBlur(_blur_img, (kx, ky), sigmaX=sigma_x, sigmaY=sigma_y, borderType=cv2.BORDER_DEFAULT)

show(gaussian_blur_k_7, title='Gaussian Blur k=7 - image')

In [ ]:
# Binary Inv Thresh 186 (SimpleThreshold)
_th_gray = ensure_gray(gaussian_blur_k_7)
thresh = 186
max_value = 255
threshold_mode = cv2.THRESH_BINARY_INV
pre_blur = odd_ksize(1, 1)
if pre_blur > 1:
    _th_gray = cv2.GaussianBlur(_th_gray, (pre_blur, pre_blur), 0)
if False:
    threshold_mode = threshold_mode | cv2.THRESH_OTSU
_, binary_inv_thresh_186_mask = cv2.threshold(_th_gray, thresh, max_value, threshold_mode)
binary_inv_thresh_186_image = cv2.cvtColor(binary_inv_thresh_186_mask, cv2.COLOR_GRAY2BGR)

show(binary_inv_thresh_186_image, title='Binary Inv Thresh 186 - image')
show(binary_inv_thresh_186_mask, title='Binary Inv Thresh 186 - mask', cmap='gray')

In [ ]:
# Morph Pass1 (Erode k=9) (BinaryMorphology)
morph_pass1_erode_k_9_mask = ensure_gray(binary_inv_thresh_186_mask)
if True:
    _, morph_pass1_erode_k_9_mask = cv2.threshold(morph_pass1_erode_k_9_mask, 127, 255, cv2.THRESH_BINARY)
if False:
    morph_pass1_erode_k_9_mask = cv2.bitwise_not(morph_pass1_erode_k_9_mask)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
anchor_x = -1
anchor_y = -1
if anchor_x >= kernel.shape[1] or anchor_x < -1:
    anchor_x = -1
if anchor_y >= kernel.shape[0] or anchor_y < -1:
    anchor_y = -1
anchor = (anchor_x, anchor_y)
morph_pass1_erode_k_9_mask = cv2.morphologyEx(
    morph_pass1_erode_k_9_mask, cv2.MORPH_ERODE, kernel, anchor=anchor, iterations=1,
    borderType=cv2.BORDER_CONSTANT, borderValue=0
)
morph_pass1_erode_k_9_image = cv2.cvtColor(morph_pass1_erode_k_9_mask, cv2.COLOR_GRAY2BGR)
morph_pass1_erode_k_9_meta = {
    'operation': 'erode',
    'kernel_shape': 'ellipse',
    'kernel_size': {'w': 9, 'h': 9},
    'iterations': 1,
    'anchor': {'x': anchor_x, 'y': anchor_y},
}

show(morph_pass1_erode_k_9_image, title='Morph Pass1 (Erode k=9) - image')
show(morph_pass1_erode_k_9_mask, title='Morph Pass1 (Erode k=9) - mask', cmap='gray')

In [ ]:
# Morph Pass2 (Erode k=1) (BinaryMorphology)
morph_pass2_erode_k_1_mask = ensure_gray(morph_pass1_erode_k_9_mask)
if True:
    _, morph_pass2_erode_k_1_mask = cv2.threshold(morph_pass2_erode_k_1_mask, 127, 255, cv2.THRESH_BINARY)
if False:
    morph_pass2_erode_k_1_mask = cv2.bitwise_not(morph_pass2_erode_k_1_mask)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 1))
anchor_x = -1
anchor_y = -1
if anchor_x >= kernel.shape[1] or anchor_x < -1:
    anchor_x = -1
if anchor_y >= kernel.shape[0] or anchor_y < -1:
    anchor_y = -1
anchor = (anchor_x, anchor_y)
morph_pass2_erode_k_1_mask = cv2.morphologyEx(
    morph_pass2_erode_k_1_mask, cv2.MORPH_ERODE, kernel, anchor=anchor, iterations=1,
    borderType=cv2.BORDER_CONSTANT, borderValue=0
)
morph_pass2_erode_k_1_image = cv2.cvtColor(morph_pass2_erode_k_1_mask, cv2.COLOR_GRAY2BGR)
morph_pass2_erode_k_1_meta = {
    'operation': 'erode',
    'kernel_shape': 'ellipse',
    'kernel_size': {'w': 1, 'h': 1},
    'iterations': 1,
    'anchor': {'x': anchor_x, 'y': anchor_y},
}

show(morph_pass2_erode_k_1_image, title='Morph Pass2 (Erode k=1) - image')
show(morph_pass2_erode_k_1_mask, title='Morph Pass2 (Erode k=1) - mask', cmap='gray')

In [ ]:
# Find External Contours (ContoursAnalysis)
find_external_contours_mask = ensure_gray(morph_pass2_erode_k_1_mask)
if True:
    _, find_external_contours_mask = cv2.threshold(find_external_contours_mask, 127, 255, cv2.THRESH_BINARY)
if False:
    find_external_contours_mask = cv2.bitwise_not(find_external_contours_mask)
_find_input = find_external_contours_mask.astype(np.int32) if cv2.RETR_EXTERNAL == cv2.RETR_FLOODFILL else find_external_contours_mask.copy()
_found = cv2.findContours(_find_input, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE, offset=(0, 0))
if len(_found) == 3:
    _, contours, hierarchy = _found
else:
    contours, hierarchy = _found
find_external_contours_image = cv2.cvtColor(find_external_contours_mask, cv2.COLOR_GRAY2BGR)
find_external_contours_detections = []
kept = 0
total_area = 0.0
rejected_area = 0
rejected_aspect = 0
for cnt in contours:
    area = float(cv2.contourArea(cnt))
    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = float(w) / float(h) if h > 0 else 9999.0
    perimeter = float(cv2.arcLength(cnt, True))
    if True and (area < 0 or area > 5972):
        rejected_area += 1
        continue
    if False and (aspect_ratio < 0.1 or aspect_ratio > 10.0):
        rejected_aspect += 1
        continue
    kept += 1
    total_area += area
    find_external_contours_detections.append({'bbox': (int(x), int(y), int(w), int(h)), 'label': 'contour', 'score': 1.0, 'extra': {'area': area, 'aspect_ratio': aspect_ratio, 'perimeter': perimeter}})
    if 'both' in ('contours', 'both'):
        cv2.drawContours(find_external_contours_image, [cnt], -1, (0, 255, 0), 2)
    if 'both' in ('boxes', 'both'):
        cv2.rectangle(find_external_contours_image, (x, y), (x + w, y + h), (0, 165, 255), 2)
find_external_contours_meta = {
    'total_contours': len(contours),
    'kept_contours': kept,
    'rejected_by_area': rejected_area,
    'rejected_by_aspect_ratio': rejected_aspect,
    'total_kept_area': total_area,
    'has_hierarchy': hierarchy is not None,
    'retrieval': 'external',
    'approx_mode': 'simple',
    'offset': {'x': 0, 'y': 0},
}

show(find_external_contours_image, title='Find External Contours - image')
show(find_external_contours_mask, title='Find External Contours - mask', cmap='gray')

In [ ]:
# Draw Boxes on RGB Crop (DrawDetections)
draw_boxes_on_rgb_crop = ensure_bgr(branch_rgb_mask_image_b).copy()
draw_detections = find_external_contours_detections or []
show_label = False
show_score = False
for det in draw_detections:
    if isinstance(det, dict):
        x, y, w, h = det.get('bbox', (0, 0, 0, 0))
        label = str(det.get('label', 'obj'))
        score = float(det.get('score', 1.0))
    else:
        x, y, w, h = getattr(det, 'bbox', (0, 0, 0, 0))
        label = str(getattr(det, 'label', 'obj'))
        score = float(getattr(det, 'score', 1.0))
    x, y, w, h = int(x), int(y), int(w), int(h)
    cv2.rectangle(draw_boxes_on_rgb_crop, (x, y), (x + w, y + h), (0, 255, 0), 2)
    text = ''
    if show_label:
        text += label
    if show_score:
        text += (' ' if text else '') + f'{score:.2f}'
    if text:
        cv2.putText(draw_boxes_on_rgb_crop, text, (x, max(10, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

show(draw_boxes_on_rgb_crop, title='Draw Boxes on RGB Crop - image')

In [ ]:
# Count Cells (ContourCount)
count_cells_image = ensure_bgr(draw_boxes_on_rgb_crop).copy()
count_cells_detections = list(find_external_contours_detections or [])
label_filter = 'contour'
exact_match = True
contour_count = 0
for _det in (find_external_contours_detections or []):
    if isinstance(_det, dict):
        _label = str(_det.get('label', ''))
    else:
        _label = str(getattr(_det, 'label', ''))
    if not label_filter:
        contour_count += 1
    elif exact_match and _label == label_filter:
        contour_count += 1
    elif (not exact_match) and label_filter.lower() in _label.lower():
        contour_count += 1
print(f'Contour count: {contour_count}')
if True:
    cv2.putText(count_cells_image, f"Detected Cells: {contour_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
count_cells_meta = {
    'contour_count': contour_count,
    'label_filter': label_filter,
    'exact_match': exact_match,
    'total_detections': len(find_external_contours_detections or []),
}

show(count_cells_image, title=f"Count Cells (count={count_cells_meta.get('contour_count', 0)})")

In [ ]:
# Compare (Mask Contours | RGB Boxes) (MergeImage)
_merge_a = ensure_bgr(find_external_contours_image)
_merge_b = ensure_bgr(count_cells_image)
merge_mode = 'hconcat'
alpha = 0.5
if merge_mode == 'choose_a':
    compare_mask_contours_rgb_boxes = _merge_a
elif merge_mode == 'choose_b':
    compare_mask_contours_rgb_boxes = _merge_b
elif merge_mode == 'alpha':
    if _merge_a.shape != _merge_b.shape:
        _merge_b = cv2.resize(_merge_b, (_merge_a.shape[1], _merge_a.shape[0]))
    compare_mask_contours_rgb_boxes = cv2.addWeighted(_merge_a, 1.0 - alpha, _merge_b, alpha, 0)
elif merge_mode == 'vconcat':
    w = min(_merge_a.shape[1], _merge_b.shape[1])
    a2 = cv2.resize(_merge_a, (w, int(_merge_a.shape[0] * w / _merge_a.shape[1])))
    b2 = cv2.resize(_merge_b, (w, int(_merge_b.shape[0] * w / _merge_b.shape[1])))
    compare_mask_contours_rgb_boxes = cv2.vconcat([a2, b2])
else:
    h = min(_merge_a.shape[0], _merge_b.shape[0])
    a2 = cv2.resize(_merge_a, (int(_merge_a.shape[1] * h / _merge_a.shape[0]), h))
    b2 = cv2.resize(_merge_b, (int(_merge_b.shape[1] * h / _merge_b.shape[0]), h))
    compare_mask_contours_rgb_boxes = cv2.hconcat([a2, b2])

show(compare_mask_contours_rgb_boxes, title='Compare (Mask Contours | RGB Boxes) - image')

In [ ]:
show(compare_mask_contours_rgb_boxes, title='Final Output')